# Track 02a — 최소 에이전트 루프

### 에이전트란?

에이전트는 **LLM을 `관찰 → 사고 → 행동(도구 호출) → 다시 관찰` 루프에 넣은 것**입니다. 모델이 한 번 답하고 끝나는 대신, 필요하면 도구를 부르고 그 결과를 다시 읽어 답이 완성될 때까지 루프를 돕니다. 이 트랙은 그 루프를 **직접 만들어 보고**, 저장소가 제공하는 에이전트로 **다시 풀어 보며** 익힙니다.

### 이 노트북에서 보여줄 것

같은 환율 질문을 **세 가지 방식**으로 풀어 봅니다.

| Session | 방식 | 무엇을 / 왜 |
|---|---|---|
| **1. 손코딩** | 프레임워크 없이 루프를 직접 구현 (표·산출물에선 `scratch` 로 표기) | 에이전트 루프의 부품(턴 예산 · 도구 호출 · 인자 파싱 · 종료 조건)을 손으로 짜서 **루프 안에서 무엇이 일어나는지** 체감합니다. |
| **2. `ToolAgent`** | 같은 문제를 저장소가 제공하는 에이전트로 | 1에서 손으로 짠 부품들(인자 검증 · 라우팅 · finalize · 중복 호출 방지 · 관측 trace)을 `ToolAgent`가 어떻게 대신 맡는지 봅니다. |
| **3. 스트리밍** | 같은 실행을 이벤트로 흘려보기 | `run_stream` 으로 한 실행을 단계 · 채널(content/reasoning) · 도구 호출 **이벤트**로 쪼개, 운영에서 쓰는 관측성의 기본 단위를 봅니다. |

> 이 비교는 같은 일을 **손으로 할 때 ↔ 프레임워크에 맡길 때** 무엇이 달라지고, `ToolAgent`가 무엇을 대신해 주는지를 보여주기 위한 것입니다.

- **구성:** 각 `Session`은 가이드(텍스트) → 코드 → 출력 해석(텍스트) 순입니다.
- **산출물:** `_out/loop_trace.json`, `_out/comparison.json`, `_out/stream_trace.jsonl`, `_out/stream.sse.txt`

In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import logging
import warnings

# (en) Quiet library logs for readable notebook output.
# (kr) 노트북 출력을 읽기 쉽게 라이브러리 로그를 줄인다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate live LLM steps on API key presence.
# (kr) API 키 유무로 라이브 LLM 단계를 가드한다.
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
TRACK02 = ROOT / "recipes" / "track02_minimum_agent_loop"
DATA = TRACK02 / "data"

base_url = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
model = (
    os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
)
client = exaone.integrations.build_llm_from_env()
print("model:", client.model)

# (en) Load the shared KRW-quoted rate table from data/ so every part uses one source.
# (kr) 모든 부분이 한 출처를 쓰도록 data/ 의 KRW 기준 환율표를 불러온다.
_RATES = json.loads((DATA / "exchange_rates.json").read_text(encoding="utf-8"))
EXCHANGE_RATES_KRW = _RATES["rates"]
AS_OF = _RATES["as_of"]


# (en) base -> quote rate, going through KRW as the pivot currency.
# (kr) base -> quote 환율. KRW 를 중심 통화로 거쳐서 계산한다.
def _rate(base: str, quote: str) -> float:
    base, quote = base.upper(), quote.upper()
    if base == "KRW" and quote == "KRW":
        return 1.0
    if base == "KRW":
        return 1.0 / EXCHANGE_RATES_KRW[quote]
    if quote == "KRW":
        return EXCHANGE_RATES_KRW[base]
    return EXCHANGE_RATES_KRW[base] / EXCHANGE_RATES_KRW[quote]


# (en) Two tool implementations shared by the hand-written loop (Part 1) and ToolAgent (Part 2).
# (kr) 손코딩 루프(1부)와 ToolAgent(2부)가 함께 쓰는 두 도구 구현이다.
def tool_exchange_rate(args: dict) -> dict:
    base = (args.get("base") or "").upper()
    quote = (args.get("quote") or "KRW").upper()
    table = set(EXCHANGE_RATES_KRW) | {"KRW"}
    if base not in table or quote not in table:
        return {"error": f"unknown currency: base={base}, quote={quote}"}
    return {
        "base": base,
        "quote": quote,
        "rate": round(_rate(base, quote), 4),
        "as_of": AS_OF,
    }


def tool_convert_money(args: dict) -> dict:
    amount = args.get("amount")
    base = (args.get("base") or "").upper()
    quote = (args.get("quote") or "KRW").upper()
    table = set(EXCHANGE_RATES_KRW) | {"KRW"}
    if not isinstance(amount, (int, float)):
        return {"error": f"amount must be a number, got {type(amount).__name__}"}
    if base not in table or quote not in table:
        return {"error": f"unknown currency: base={base}, quote={quote}"}
    converted = float(amount) * _rate(base, quote)
    return {
        "amount": amount,
        "base": base,
        "quote": quote,
        "converted": round(converted, 2),
        "rate_used": round(_rate(base, quote), 4),
    }


# (en) OpenAI-compatible tool schemas. The hand-written loop and ToolRegistry both consume these.
# (kr) OpenAI 호환 도구 스키마. 손코딩 루프와 ToolRegistry 가 모두 이걸 사용한다.
EXCHANGE_RATE_SCHEMA = {
    "type": "function",
    "function": {
        "name": "exchange_rate",
        "description": "Get the exchange rate from base currency to quote currency. Use when the user asks for a rate without a specific amount.",
        "parameters": {
            "type": "object",
            "required": ["base"],
            "properties": {
                "base": {
                    "type": "string",
                    "enum": ["USD", "EUR", "JPY", "CNY", "GBP", "KRW"],
                },
                "quote": {
                    "type": "string",
                    "enum": ["USD", "EUR", "JPY", "CNY", "GBP", "KRW"],
                    "default": "KRW",
                },
            },
            "additionalProperties": False,
        },
    },
}
CONVERT_MONEY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "convert_money",
        "description": "Convert a specific amount of base currency into quote currency. Use whenever the user mentions a numeric amount.",
        "parameters": {
            "type": "object",
            "required": ["amount", "base"],
            "properties": {
                "amount": {"type": "number", "minimum": 0},
                "base": {
                    "type": "string",
                    "enum": ["USD", "EUR", "JPY", "CNY", "GBP", "KRW"],
                },
                "quote": {
                    "type": "string",
                    "enum": ["USD", "EUR", "JPY", "CNY", "GBP", "KRW"],
                    "default": "KRW",
                },
            },
            "additionalProperties": False,
        },
    },
}
TOOL_SCHEMAS = [EXCHANGE_RATE_SCHEMA, CONVERT_MONEY_SCHEMA]

SYSTEM_PROMPT = (
    "You are a careful Korean-speaking assistant for currency questions. "
    "Use the provided tools whenever the user mentions currencies or amounts; "
    "do not guess. After all tools have answered, reply in Korean with one short sentence "
    "including each computed amount and the as-of date when relevant."
)

out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)

print("smoke test (순수 함수라 API 불필요):")
print("  exchange_rate:", tool_exchange_rate({"base": "USD", "quote": "KRW"}))
print(
    "  convert_money:",
    tool_convert_money({"amount": 100, "base": "USD", "quote": "KRW"}),
)
print("준비 완료. HAS_API =", HAS_API)

**출력 해석:** `model:`과 경로 변수가 보이면 이 노트북에서 쓸 client와 DATA가 준비된 것입니다.


## Session 1. 손코딩 에이전트 루프

**테스트 시나리오** — `INPUTS` 3건(환율 질의). 정답 여부보다 **turn·tool_calls·stop_reason** 을 중심으로 봅니다.

| id | 질문 요지 | 기대 패턴 |
|---|---|---|
| `single` | 100달러→원화 | 도구 1회 |
| `two_calls` | USD+EUR 합산 | 도구 2회 |
| `no_tool` | 환율 개념 설명 | 도구 0회 |

프레임워크 없이 `run_agent(...)` 함수 하나에 루프 전체를 담아 봅니다.


In [ ]:
# (en) One run's outcome. We keep the full per-turn trace so Part 2 can compare line by line.
# (kr) 한 실행의 결과. 2부에서 줄 단위 비교를 하려고 턴별 trace 를 통째로 보관한다.
@dataclass
class AgentRun:
    final_answer: str
    stop_reason: str
    turns_used: int
    tool_invocations: int
    total_latency_ms: float
    turns: list = field(default_factory=list)
    error: str = None


# (en) TOOL_IMPLS is the from-scratch "registry": tool name -> python callable.
# (kr) TOOL_IMPLS 는 손코딩 "레지스트리"다. 도구 이름 -> 파이썬 함수 매핑이며, 2부의 ToolRegistry 가 같은 일을 클래스로 감싼다.
TOOL_IMPLS = {"exchange_rate": tool_exchange_rate, "convert_money": tool_convert_money}


def run_agent(
    user_query: str,
    *,
    system_prompt: str = SYSTEM_PROMPT,
    max_turns: int = 5,
    max_consecutive_tool_errors: int = 3,
    enable_thinking: bool = False,
    max_new_tokens: int = 512,
) -> AgentRun:
    messages = [
        exaone.llm.ExaoneMessage(role="system", content=system_prompt),
        exaone.llm.ExaoneMessage(role="user", content=user_query),
    ]
    opts = exaone.llm.ExaoneGenerateOptions(
        enable_thinking=enable_thinking,
        max_new_tokens=max_new_tokens,
        tools=TOOL_SCHEMAS,
    )

    turns, tool_invocations, consecutive_tool_errors, total_latency_ms = [], 0, 0, 0.0
    stop_reason, final_answer = "max_turns", ""

    for turn_idx in range(1, max_turns + 1):
        t0 = time.monotonic()
        try:
            resp = client.chat(messages, options=opts)
        except Exception as exc:
            return AgentRun(
                final_answer,
                "llm_error",
                turn_idx - 1,
                tool_invocations,
                total_latency_ms,
                turns,
                f"{type(exc).__name__}: {exc}",
            )
        latency = (time.monotonic() - t0) * 1000
        total_latency_ms += latency

        messages.append(
            exaone.llm.ExaoneMessage(
                role="assistant", content=resp.content or "", tool_calls=resp.tool_calls
            )
        )
        turn_record = {
            "turn": turn_idx,
            "llm_latency_ms": round(latency, 1),
            "content_preview": (resp.content or "")[:120],
            "tool_calls": [],
            "finish_reason": resp.finish_reason,
        }

        if not resp.tool_calls:
            final_answer = resp.content or ""
            stop_reason = "no_tool_calls"
            turns.append(turn_record)
            break

        any_error_this_turn = False
        for tc in resp.tool_calls:
            tc_id = tc.get("id") or ""
            fn = tc.get("function") or {}
            name = (fn.get("name") or "").strip()
            raw_args = fn.get("arguments") or "{}"

            args, parse_err = {}, None
            try:
                args = (
                    json.loads(raw_args)
                    if isinstance(raw_args, str)
                    else dict(raw_args)
                )
            except Exception as e:
                parse_err = f"{type(e).__name__}: {e}"

            if parse_err:
                result = {"error": f"arguments JSON parse failed: {parse_err}"}
                any_error_this_turn = True
            elif name not in TOOL_IMPLS:
                result = {"error": f"unknown tool: {name}"}
                any_error_this_turn = True
            else:
                try:
                    result = TOOL_IMPLS[name](args)
                    if isinstance(result, dict) and "error" in result:
                        any_error_this_turn = True
                except Exception as e:
                    result = {"error": f"{type(e).__name__}: {e}"}
                    any_error_this_turn = True

            tool_invocations += 1
            turn_record["tool_calls"].append(
                {
                    "tool_call_id": tc_id,
                    "name": name,
                    "arguments": args,
                    "result": result,
                }
            )
            messages.append(
                exaone.llm.ExaoneMessage(
                    role="tool",
                    tool_call_id=tc_id,
                    name=name,
                    content=json.dumps(result, ensure_ascii=False),
                )
            )

        turns.append(turn_record)
        consecutive_tool_errors = (
            consecutive_tool_errors + 1 if any_error_this_turn else 0
        )
        if consecutive_tool_errors >= max_consecutive_tool_errors:
            stop_reason = "too_many_tool_errors"
            final_answer = (resp.content or "").strip() or final_answer
            break
    else:
        # (en) for-else: this runs only when the loop finished without break (hit max_turns).
        # (kr) for-else: break 없이 루프를 끝까지 돌았을 때만(=max_turns 도달) 실행된다.
        final_answer = (resp.content or "").strip() or final_answer

    return AgentRun(
        final_answer, stop_reason, len(turns), tool_invocations, total_latency_ms, turns
    )


print("run_agent 정의 완료. 손코딩 도구:", list(TOOL_IMPLS))

**출력 해석:** `run_agent 정의 완료. 손코딩 도구:` 가 보이면 이 단계는 통과입니다.


### Session 1-1. INPUTS 3건 실행

**하는 일:** `INPUTS` 3건을 `run_agent` 로 돌려 turn·도구 호출·종료 사유를 봅니다.

**정상:** `시나리오:` 줄 + 각 입력의 `── <id>: '<질문>' ──` 헤더 3건

**의미:** Session 2의 `ToolAgent`와 **같은 입력**으로 비교할 기준선입니다.

In [ ]:
print("시나리오: 환율 agent 입력 3건 (Track 02 INPUTS)")
INPUTS = [
    {"id": "single", "query": "100달러는 원화로 얼마야?"},
    {"id": "two_calls", "query": "100달러랑 50유로는 합쳐서 원화로 얼마야?"},
    {"id": "no_tool", "query": "환율이라는 개념을 한 문장으로 설명해줘."},
]

runs = []
for inp in INPUTS:
    print(f"── {inp['id']}: {inp['query']!r} ──")
    run = run_agent(inp["query"], max_turns=4)
    print(
        f"   turns={run.turns_used}  tool_calls={run.tool_invocations}  "
        f"stop={run.stop_reason}  latency={run.total_latency_ms:.0f}ms"
    )
    print(f"   answer: {run.final_answer[:160]}")
    if run.error:
        print(f"   error : {run.error}")
    runs.append({"input": inp, "run": run})
    print()

**출력 해석:** 입력 3건마다 `── <id>: '<질문>' ──` 헤더와 그 아래 `turns=… tool_calls=… stop=… latency=…ms`·`answer:` 줄이 나오면 정상입니다.

In [ ]:
print(f"{'id':<12}{'turns':>6}{'tool_calls':>12}{'latency_ms':>13}  stop_reason")
print("-" * 60)
for r in runs:
    run = r["run"]
    print(
        f"{r['input']['id']:<12}{run.turns_used:>6}{run.tool_invocations:>12}"
        f"{run.total_latency_ms:>13.0f}  {run.stop_reason}"
    )

print("\n['two_calls' 의 턴별 도구 호출 상세]")
two_calls = next(r["run"] for r in runs if r["input"]["id"] == "two_calls")
for t in two_calls.turns:
    print(
        f"  turn {t['turn']}  llm_latency={t['llm_latency_ms']}ms  tool_calls={len(t['tool_calls'])}"
    )
    for tc in t["tool_calls"]:
        ok = (
            "ERR"
            if isinstance(tc["result"], dict) and "error" in tc["result"]
            else "OK "
        )
        print(f"    [{ok}] {tc['name']}({tc['arguments']}) → {tc['result']}")

**출력 해석:** `id`별 `turns`·`tool_calls`·`stop_reason` 표와 `two_calls`의 턴별 도구 호출 상세가 나오면 정상입니다.


### Session 1-2. 안전장치 검증 — `max_turns` 와 잘못된 도구 인자

**하는 일:** 안전장치 검증 — `max_turns` 와 잘못된 도구 인자.

**정상:** `[max_turns=1]`·`[미지원 통화 → 복구]` 블록과 `turn N [ERR]/[OK ] …` 줄이 보임

**의미:** 운영에서 실제로 일어나는 두 상황을 일부러 만들어 봅니다. `max_turns=1`은 도구를 호출했지만 *최종 답을 만들 기회는 주지 않는* 상황입니다.

In [ ]:
budget_run = None
recover_run = None
LOOSE_SYS = (
    "You are a currency assistant. Try to call exchange_rate or convert_money for any currency the user mentions, "
    "even rare ones like RUB, BRL, INR. If the tool returns an error, gracefully tell the user in Korean."
)

budget_run = run_agent("100달러랑 50유로는 합쳐서 원화로 얼마야?", max_turns=1)
print("[max_turns=1]")
print(f"  stop_reason = {budget_run.stop_reason}")
print(f"  turns_used  = {budget_run.turns_used}")
print(f"  tool_calls  = {budget_run.tool_invocations}")
print(f"  final_answer (budget=1 에선 비어도 정상): {budget_run.final_answer!r}")

recover_run = run_agent(
    "100루블을 원화로 바꿔줘.", system_prompt=LOOSE_SYS, max_turns=4
)
print("\n[미지원 통화 → 복구]")
print(f"  stop_reason = {recover_run.stop_reason}")
print(f"  turns_used  = {recover_run.turns_used}")
print(f"  tool_calls  = {recover_run.tool_invocations}")
print(f"  final_answer: {recover_run.final_answer[:200]}")
for t in recover_run.turns:
    for tc in t["tool_calls"]:
        mark = (
            "ERR"
            if isinstance(tc["result"], dict) and "error" in tc["result"]
            else "OK "
        )
        print(
            f"    turn {t['turn']} [{mark}] {tc['name']}({tc['arguments']}) → {tc['result']}"
        )

**출력 해석:** `[max_turns=1]` 블록과 `[미지원 통화 → 복구]` 블록이 각각 `stop_reason / turns_used / tool_calls / final_answer` 로 나오고, 복구 블록에 `[ERR]`/`[OK ]` 도구 호출 줄이 보이면 정상입니다.

**여기서 보이는 한계 (의도된 단순화):**

- `max_turns=1`로 답이 비면, 운영에서는 *finalize 호출* 을 한 번 더 보내야 합니다 — 이 일을 `ToolAgent`의 `request_final_turn`이 자동으로 처리합니다.
- 미지원 통화에서 모델이 `tool error`를 보고 한국어로 설명하면 정상입니다. 같은 도구를 계속 다시 부르면 `consecutive_tool_errors`가 올라가 안전장치가 걸립니다.

### Session 1-3. 산출물 — `loop_trace.json`

기본 입력 3건과 안전장치 2건, 총 5건의 trace 를 한 JSON 으로 저장합니다. Session 2에서 `ToolAgent` 결과와 비교할 때의 *기준선* 이 됩니다.


In [ ]:
trace_payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": client.model,
    "tools": TOOL_SCHEMAS,
    "system_prompt": SYSTEM_PROMPT,
    "primary_runs": [{"input": r["input"], "run": asdict(r["run"])} for r in runs],
    "safeguards": {
        "max_turns_1": {
            "input": "100달러랑 50유로는 합쳐서 원화로 얼마야?",
            "run": asdict(budget_run) if budget_run else None,
        },
        "unknown_currency": {
            "input": "100루블을 원화로 바꿔줘.",
            "system_prompt": LOOSE_SYS,
            "run": asdict(recover_run) if recover_run else None,
        },
    },
}
out_path = out_dir / "loop_trace.json"
out_path.write_text(
    json.dumps(trace_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("saved:", out_path.resolve(), f"({out_path.stat().st_size} bytes)")

**출력 해석:** `saved:` 가 보이면 이 단계는 통과입니다.


## Session 2. 같은 문제를 `ToolAgent` 로

Session 1의 손코딩 루프를 비교 대상으로 두고, 같은 도구·같은 시스템 프롬프트·같은 입력을 `exaone.agents.ToolAgent`로 다시 풉니다.

| 책임 | Session 1 (손코딩) | `ToolAgent` |
|---|---|---|
| 도구 정의 | `TOOL_SCHEMAS` + `TOOL_IMPLS` | `Tool` + `ToolRegistry` (스키마·인자 검증 포함) |
| while 루프 | `for turn in range(max_turns)` | `BaseAgent` 내부 루프 재사용 |
| 인자 파싱 | `json.loads` + 가드 | 레지스트리가 내부에서 처리 |
| 연속 에러 한도 | `max_consecutive_tool_errors` | 동일 (기본 3) |
| **thinking 채널** | 미사용 | `ThinkingRouter`가 입력별 자동 결정 |
| **최종 답 강제** | 없음 (빈 답 가능) | `request_final_turn` (finalize 단계) |
| **중복 호출 방지** | 없음 | `NextStepPlanner` + ledger |
| **trace 식별** | 자체 dict | 관측성 필드 (Track 07 과 연결) |

라우터와 플래너의 효과가 보이도록 **두 변형** 으로 비교합니다.

- **scratch** — Session 1의 `runs`를 그대로 사용 (재실행하지 않아 같은 결과를 기준으로 비교).
- **tool_agent_full** — `ToolAgent(router on, planner on)`.
- **tool_agent_minimal** — `ToolAgent(router off, planner off)` — 손코딩과 가장 가까운 변형.


In [ ]:
# (en) Same two tools, now wrapped as exaone.tools.Tool inside a ToolRegistry.
# (kr) 같은 두 도구를 이번엔 exaone.tools.Tool 로 감싸 ToolRegistry 에 넣는다.
def build_registry() -> "exaone.tools.ToolRegistry":
    reg = exaone.tools.ToolRegistry()
    reg.register(
        exaone.tools.Tool(
            name="exchange_rate",
            schema=EXCHANGE_RATE_SCHEMA,
            execute=lambda a: tool_exchange_rate(a),
        )
    )
    reg.register(
        exaone.tools.Tool(
            name="convert_money",
            schema=CONVERT_MONEY_SCHEMA,
            execute=lambda a: tool_convert_money(a),
        )
    )
    return reg


agent_full = exaone.agents.ToolAgent(
    tool_registry=build_registry(),
    system_prompt=SYSTEM_PROMPT,
    max_turns=5,
    use_thinking_router=True,
    use_next_step_planner=True,
)
agent_minimal = exaone.agents.ToolAgent(
    tool_registry=build_registry(),
    system_prompt=SYSTEM_PROMPT,
    max_turns=5,
    use_thinking_router=False,
    use_next_step_planner=False,
)


# (en) Run one ToolAgent variant and flatten its metadata into a comparable row.
# (kr) ToolAgent 한 변형을 실행하고 metadata 를 비교 가능한 한 행으로 펼친다.
def run_variant(agent, query: str) -> dict:
    ctx = exaone.agents.AgentContext(query=query)
    t0 = time.monotonic()
    try:
        result = agent.run(ctx, llm=client, verbose=False)
    except Exception as e:
        return {
            "final_answer": "",
            "turns_used": 0,
            "tool_invocations": 0,
            "total_latency_ms": (time.monotonic() - t0) * 1000,
            "stop_reason": "exception",
            "llm_calls_count": 0,
            "error": f"{type(e).__name__}: {e}",
        }
    latency = (time.monotonic() - t0) * 1000
    meta = dict(result.metadata or {})
    return {
        "final_answer": result.content or "",
        "turns_used": meta.get("turns_used", 0),
        "tool_invocations": meta.get("tool_invocations", 0),
        "total_latency_ms": round(latency, 1),
        "stop_reason": meta.get("enrich_stop_reason"),
        "llm_calls_count": len(meta.get("llm_calls") or []),
        "error": None if result.success else result.error,
    }


print("exaone.agents.ToolAgent 두 변형 준비 완료 (full / minimal).")

**출력 해석:** `exaone.agents.ToolAgent 두 변형 준비 완료 (full / minimal).` 이 보이면 이 단계는 통과입니다.


### Session 2-1. 3-way 실행

**하는 일:** 3-way 실행.

**정상:** 각 `── <id>: '<질문>' ──` 헤더 아래 `tool_agent_full …`·`tool_agent_minimal …` 줄이 보임

**의미:** 같은 입력 3개를 두 변형으로 돌립니다. `scratch` 행은 Session 1의 `runs` 에서 가져오므로, 달라진 조건은 *루프 구현* 하나뿐입니다.

In [ ]:
variant_results = {"tool_agent_full": {}, "tool_agent_minimal": {}}
for inp in INPUTS:
    print(f"── {inp['id']}: {inp['query']!r} ──")
    for label, agent in [
        ("tool_agent_full", agent_full),
        ("tool_agent_minimal", agent_minimal),
    ]:
        r = run_variant(agent, inp["query"])
        variant_results[label][inp["id"]] = r
        print(
            f"  {label:<20} turns={r['turns_used']}  tools={r['tool_invocations']}  "
            f"llm_calls={r['llm_calls_count']}  lat={r['total_latency_ms']:.0f}ms  stop={r['stop_reason']}"
        )
    print()

**출력 해석:** 입력 3건마다 `── <id>: '<질문>' ──` 헤더 아래 `tool_agent_full …`·`tool_agent_minimal …` 두 줄이 `turns= tools= llm_calls= lat= stop=` 형태로 나오면 정상입니다.

In [ ]:
# (en) Build a scratch comparison row directly from Part 1's in-memory runs (no file reload).
# (kr) 1부의 메모리 runs 에서 곧장 scratch 비교 행을 만든다 (파일 재로드 없음).
def _scratch_row(inp_id: str) -> dict:
    run = next(r["run"] for r in runs if r["input"]["id"] == inp_id)
    return {
        "final_answer": run.final_answer,
        "turns_used": run.turns_used,
        "tool_invocations": run.tool_invocations,
        "total_latency_ms": run.total_latency_ms,
        "stop_reason": run.stop_reason,
        "llm_calls_count": run.turns_used,
    }


# (en) Mechanical semantic check: the sentence may differ but the key numbers must appear.
# (kr) 기계적 의미 검증. 문장은 달라도 핵심 수치는 들어 있어야 한다.
EXPECTATIONS = {
    "single": {"must_contain_any": ["138,050", "138050"], "must_have_tools": True},
    "two_calls": {"must_contain_any": ["74,510", "74510"], "must_have_tools": True},
    "no_tool": {"must_have_tools": False},
}


def _check(answer: str, used_tools: int, exp: dict):
    fails = []
    if exp.get("must_have_tools") is True and used_tools == 0:
        fails.append("expected at least one tool call, got 0")
    if exp.get("must_have_tools") is False and used_tools > 0:
        fails.append(f"expected zero tool calls, got {used_tools}")
    any_tokens = exp.get("must_contain_any") or []
    if any_tokens and not any(tok in answer for tok in any_tokens):
        fails.append(f"none of {any_tokens} in answer")
    return (not fails), fails


table_rows = []
verdicts = []
print(
    f"{'input':<11}{'impl':<22}{'turns':>6}{'tools':>6}{'llm':>5}{'latency_ms':>12}  pass"
)
print("-" * 78)
for inp in INPUTS:
    rows = [
        ("scratch", _scratch_row(inp["id"])),
        ("tool_agent_full", variant_results["tool_agent_full"][inp["id"]]),
        ("tool_agent_minimal", variant_results["tool_agent_minimal"][inp["id"]]),
    ]
    for label, r in rows:
        ok, fails = _check(
            r["final_answer"], r["tool_invocations"], EXPECTATIONS[inp["id"]]
        )
        verdicts.append(
            {"input_id": inp["id"], "impl": label, "pass": ok, "fails": fails, **r}
        )
        table_rows.append({"input_id": inp["id"], "impl": label, **r})
        print(
            f"{inp['id']:<11}{label:<22}{r['turns_used']:>6}{r['tool_invocations']:>6}"
            f"{r['llm_calls_count']:>5}{r['total_latency_ms']:>12.0f}  {'PASS' if ok else 'FAIL'}"
        )
    print()

**출력 해석:** `input`·`impl`별 요약 표가 나오고 각 행에 `PASS` 또는 `FAIL` 이 표시되면 정상입니다. `scratch` 행은 Session 1의 `runs` 에서 가져오므로, 달라진 조건은 *루프 구현* 하나뿐입니다.


### Session 2-2. 일부러 어려운 입력 — `ToolAgent` 의 진가

**하는 일:** 일부러 어려운 입력 — `ToolAgent` 의 진가.

**정상:** 두 변형마다 `turns= tools= llm_calls= lat=` 줄과 `answer:` 가 보임

**의미:** `two_calls`보다 한 단계 어려운 입력입니다. 먼저 `convert_money(100, USD, KRW)`를 호출하고, 그 결과의 *절반* 을 다시 `convert_money(amount, KRW, EUR)`로 넘겨야 합니다. 두 번째 인자가 첫 번째 결과에 의존하므로, 손코딩 루프라면 보통 다음 턴까지 기다려야 합니다. `ToolAgent`의 `NextStepPlanner`가 이런 의존 관계에서 루프 진행을 어떻게 돕는지 확인합니다.

In [ ]:
CHAIN_QUERY = "100달러를 원화로 바꾼 다음, 그 금액의 절반을 다시 유로로 바꿔줘."
chain_results = {}
for label, agent in [
    ("tool_agent_full", agent_full),
    ("tool_agent_minimal", agent_minimal),
]:
    r = run_variant(agent, CHAIN_QUERY)
    chain_results[label] = r
    print(
        f"{label:<22} turns={r['turns_used']}  tools={r['tool_invocations']}  "
        f"llm_calls={r['llm_calls_count']}  lat={r['total_latency_ms']:.0f}ms"
    )
    print(f"  answer: {(r['final_answer'] or '')[:200]}\n")

**출력 해석:** 두 변형마다 `turns= tools= llm_calls= lat=` 요약 줄과 한국어 `answer:` 가 나오면 정상입니다.

In [ ]:
impls = ("scratch", "tool_agent_full", "tool_agent_minimal")
payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": client.model,
    "system_prompt": SYSTEM_PROMPT,
    "inputs": INPUTS,
    "implementations": list(impls),
    "primary_table": verdicts,
    "summary": {
        "pass_rate_by_impl": {
            impl: round(
                sum(1 for v in verdicts if v["impl"] == impl and v["pass"])
                / max(1, sum(1 for v in verdicts if v["impl"] == impl)),
                3,
            )
            for impl in impls
        },
        "avg_latency_ms_by_impl": {
            impl: round(
                sum(v["total_latency_ms"] for v in verdicts if v["impl"] == impl)
                / max(1, sum(1 for v in verdicts if v["impl"] == impl)),
                1,
            )
            for impl in impls
        },
    },
    "dependent_chain": {"input": CHAIN_QUERY, "results": chain_results},
}
out_path = out_dir / "comparison.json"
out_path.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print("saved:", out_path.resolve())
print("\n[summary]")
print(json.dumps(payload["summary"], ensure_ascii=False, indent=2))

**출력 해석:** `saved:` 와 `[summary]` 가 보이면 이 단계는 통과입니다.


## Session 3. 스트리밍 — 같은 실행을 채널별로 흘리기

`ToolAgent.run()`은 최종 답을 한 번에 반환합니다.

| event | 의미 | 본 노트북에서 |
|---|---|---|
| `run_start` / `run_end` | 한 실행의 시작/끝 | 헤더 · 최종 답 |
| `phase_start` / `phase_end` | preflight → enrich → finalize 단계 경계 | 채널 라벨 |
| `planner_end` | `NextStepPlanner`의 결정 | 디버그 라인 |
| `turn_start` | enrich 한 턴 시작 | `── turn N ──` |
| `llm_delta` | LLM 출력 청크 (`channel ∈ {content, reasoning}`) | 채널별 색·라벨 |
| `llm_end` | 한 LLM 호출 끝 | usage·latency |
| `tool_start` / `tool_end` | 한 도구 실행의 시작/끝 | 도구명·인자·결과 |
| `error` | 에러 | stderr |


In [ ]:
# (en) ANSI colors so Jupyter and a real terminal render the same channel labels.
# (kr) Jupyter 와 실제 터미널이 같은 채널 라벨을 그리도록 ANSI 색을 쓴다.
ANSI = {
    "dim": "\033[2m",
    "bold": "\033[1m",
    "cyan": "\033[36m",
    "yellow": "\033[33m",
    "green": "\033[32m",
    "red": "\033[31m",
    "reset": "\033[0m",
}


# (en) One sink that turns each AgentEvent into a labeled console line; reasoning is dimmed/buffered.
# (kr) 각 AgentEvent 를 라벨 붙은 콘솔 한 줄로 바꾸는 sink. reasoning 채널은 흐리게·버퍼링한다.
def render_event(ev, *, reasoning_buf: list, content_first: list) -> None:
    t = ev.type
    p = ev.payload or {}

    if t == "run_start":
        has = "tools=on" if p.get("has_tools") else "tools=off"
        print(
            f"{ANSI['bold']}┏━ run_start ({has}, stream_llm={p.get('stream_llm')}, "
            f"stream_reasoning={p.get('stream_enrich_reasoning')}){ANSI['reset']}"
        )
    elif t == "phase_start":
        print(f"{ANSI['cyan']}┃ ▸ phase_start: {p.get('phase')}{ANSI['reset']}")
    elif t == "phase_end":
        info = ", ".join(f"{k}={v}" for k, v in p.items() if k != "phase")
        print(
            f"{ANSI['cyan']}┃ ◂ phase_end:   {p.get('phase')}  {ANSI['dim']}{info}{ANSI['reset']}"
        )
    elif t == "planner_end":
        print(
            f"{ANSI['yellow']}┃   planner_end: kind={p.get('kind')} "
            f"answerable={p.get('answerable')} key={p.get('tool_agent_key')}{ANSI['reset']}"
        )
    elif t == "turn_start":
        print(
            f"\n{ANSI['bold']}┃ ── turn {ev.turn} [{p.get('phase', '-')}] ──{ANSI['reset']}"
        )
        content_first[0] = True
    elif t == "llm_delta":
        ch, text = p.get("channel"), p.get("text", "")
        if ch == "content":
            if content_first[0]:
                sys.stdout.write(f"{ANSI['green']}┃ ✎ ")
                content_first[0] = False
            sys.stdout.write(text)
            sys.stdout.flush()
        elif ch == "reasoning":
            reasoning_buf.append(text)
    elif t == "llm_end":
        if reasoning_buf:
            joined = "".join(reasoning_buf).strip().replace("\n", " ")
            if len(joined) > 120:
                joined = joined[:117] + "..."
            sys.stdout.write(f"\n{ANSI['dim']}┃ │ reasoning: {joined}{ANSI['reset']}\n")
            reasoning_buf.clear()
        elif not content_first[0]:
            sys.stdout.write(f"{ANSI['reset']}\n")
        print(
            f"{ANSI['dim']}┃   llm_end: content_len={p.get('content_length')} "
            f"reasoning_len={p.get('reasoning_length')} has_tool_calls={p.get('has_tool_calls')} "
            f"latency={p.get('latency_ms')}ms{ANSI['reset']}"
        )
    elif t == "tool_start":
        print(
            f"{ANSI['yellow']}┃ ▶ tool_start: {p.get('tool')}({p.get('args_preview')}){ANSI['reset']}"
        )
    elif t == "tool_end":
        print(
            f"{ANSI['yellow']}┃ ◀ tool_end:   {p.get('tool')} → {p.get('result_preview')}{ANSI['reset']}"
        )
    elif t == "run_end":
        final = (p.get("final_content") or "").replace("\n", " ")
        if len(final) > 200:
            final = final[:197] + "..."
        err = p.get("error")
        marker = f"{ANSI['red']}error={err}{ANSI['reset']}" if err else "ok"
        print(
            f"{ANSI['bold']}┗━ run_end ({marker}) turns={p.get('turns_used')} final={final!r}{ANSI['reset']}"
        )
    elif t == "error":
        print(f"{ANSI['red']}┃ ! error: {p.get('message')}{ANSI['reset']}")
    else:
        print(f"{ANSI['dim']}┃ ? unknown {t}: {p}{ANSI['reset']}")


print("render_event 정의 완료.")

**출력 해석:** `render_event 정의 완료.` 가 보이면 이 단계는 통과입니다.


### Session 3-1. 라이브 실행 — `stream_llm=True`, `stream_enrich_reasoning=True`

**하는 일:** 라이브 실행 — `stream_llm=True, stream_enrich_reasoning=True`.

**정상:** `┏━ run_start …` 부터 `┗━ run_end …` 까지 색 입힌 이벤트 줄이 보임

**의미:** 같은 입력을 한 번 실행하면서 화면에는 콘솔 sink 로 흘려보내고, 동시에 모든 이벤트를 `events_recorded`와 `sse_lines` 양쪽에 모아 산출물로 저장합니다. 노트북에서는 ANSI 색이 그대로 보입니다.

In [ ]:
QUERY = "100달러랑 50유로는 합쳐서 원화로 얼마야?"
events_recorded = []
sse_lines = []
event_counts = {}
char_counts = {"content": 0, "reasoning": 0}

agent_stream = exaone.agents.ToolAgent(
    tool_registry=build_registry(),
    system_prompt=SYSTEM_PROMPT,
    max_turns=4,
    use_thinking_router=True,
    use_next_step_planner=True,
)
ctx = exaone.agents.AgentContext(query=QUERY)
reasoning_buf, content_first = [], [True]
print(f"{ANSI['bold']}query:{ANSI['reset']} {QUERY}\n")
t0 = time.monotonic()
try:
    for ev in agent_stream.run_stream(
        ctx, llm=client, verbose=False, stream_llm=True, stream_enrich_reasoning=True
    ):
        events_recorded.append(ev)
        sse_lines.append(exaone.agents.agent_event_to_sse(ev))
        event_counts[ev.type] = event_counts.get(ev.type, 0) + 1
        if ev.type == "llm_delta":
            ch = ev.payload.get("channel")
            if ch in char_counts:
                char_counts[ch] += len(ev.payload.get("text", ""))
        render_event(ev, reasoning_buf=reasoning_buf, content_first=content_first)
except Exception as e:
    print(f"\n{ANSI['red']}EXCEPTION: {type(e).__name__}: {e}{ANSI['reset']}")
elapsed = (time.monotonic() - t0) * 1000
print(
    f"\n{ANSI['dim']}stream finished in {elapsed:.0f} ms — {len(events_recorded)} events{ANSI['reset']}"
)

**출력 해석:** `┏━ run_start …` 부터 `┗━ run_end …` 까지 색 입힌 이벤트 줄이 흐르고, 끝에 `stream finished in … ms — N events` 가 나오면 정상입니다.

### Session 3-2. 이벤트 통계 — 대략 이런 비율이 정상

**하는 일:** 이벤트 통계 — 대략 이런 비율이 정상.

**정상:** 이벤트별 `count` 표와 `chars streamed …` 줄, 이어서 `invariants` 5개가 보임

**의미:** 추론 모드에서는 깊은 답이 필요할 때 reasoning 채널과 최종 답(content)을 함께 받습니다.

In [ ]:
print(f"{'event':<14} count")
print("-" * 24)
for k in sorted(event_counts):
    print(f"{k:<14} {event_counts[k]}")
print(
    f"\nchars streamed   content={char_counts['content']}  reasoning={char_counts['reasoning']}"
)

invariants = {
    "exactly_one_run_start": event_counts.get("run_start", 0) == 1,
    "exactly_one_run_end": event_counts.get("run_end", 0) == 1,
    "tool_start_end_paired": event_counts.get("tool_start", 0)
    == event_counts.get("tool_end", 0),
    "phase_start_end_paired": event_counts.get("phase_start", 0)
    == event_counts.get("phase_end", 0),
    "no_errors": event_counts.get("error", 0) == 0,
}
print("\ninvariants")
for k, v in invariants.items():
    print(f"  [{'OK  ' if v else 'FAIL'}] {k}")

**출력 해석:** 이벤트 종류별 `count` 표와 `chars streamed content=… reasoning=…` 줄에 이어, `invariants` 5개가 모두 `[OK  ]` 면 정상입니다.

### Session 3-3. SSE 메시지 형식 확인

**하는 일:** SSE 메시지가 어떤 형식으로 만들어지는지 확인합니다.

**정상:** 에러 없이 예시 출력이 나옴

**의미:** `agent_event_to_sse(ev)`는 `event: <type>\ndata: <json>\n\n` 형식의 메시지 한 덩어리를 만듭니다. 줄바꿈은 `\n`, `data:`는 JSON 한 줄, 이벤트 사이의 빈 줄 하나가 메시지 경계입니다.


In [ ]:
print("first 3 SSE messages (verbatim):\n")
for s in sse_lines[:3]:
    sys.stdout.write(s)
print("---\n\nfirst content-channel delta (verbatim):\n")
for s in sse_lines:
    if '"channel": "content"' in s:
        sys.stdout.write(s)
        break

**출력 해석:** 에러 없이 예시 SSE 메시지가 출력되면 이 단계는 통과입니다.


### Session 3-4. 산출물 — `stream_trace.jsonl` · `stream.sse.txt`

**하는 일:** 앞 단계 결과를 `stream_trace.jsonl`과 `stream.sse.txt`에 저장합니다.

**정상:** `saved:` 아래에 파일 경로 2개가 보임

**의미:** `stream_trace.jsonl`은 한 줄에 `AgentEvent` 하나를 담아 재실행·리플레이에 쓰고, `stream.sse.txt`는 같은 실행의 SSE 덤프로 백엔드에서 그대로 전송할 수 있는 형식입니다.


In [ ]:
jsonl_path = out_dir / "stream_trace.jsonl"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for ev in events_recorded:
        fh.write(json.dumps(ev.to_dict(), ensure_ascii=False))
        fh.write("\n")
sse_path = out_dir / "stream.sse.txt"
sse_path.write_text("".join(sse_lines), encoding="utf-8")
print("saved:")
print(
    "  ",
    jsonl_path.resolve(),
    f"({jsonl_path.stat().st_size} bytes, {len(events_recorded)} events)",
)
print(
    "  ",
    sse_path.resolve(),
    f"({sse_path.stat().st_size} bytes, {len(sse_lines)} messages)",
)

**출력 해석:** `saved:` 아래에 `stream_trace.jsonl`과 `stream.sse.txt` 경로가 보이면 이 단계는 통과입니다.


## 같은 SSE 를 독립 실행 서버로

위 콘솔 sink 와 같은 이벤트 흐름을 노트북 밖 FastAPI 서버에서도 돌릴 수 있도록 옆 폴더에 예제 코드를 두었습니다: [`streaming_demo/app.py`](./streaming_demo/app.py).

```bash
uvicorn recipes.track02_minimum_agent_loop.streaming_demo.app:app --reload --port 8765
curl -N "http://127.0.0.1:8765/v1/agent/stream?query=hello&stream_llm=false"
```

API 키가 없으면 프로세스 내부 mock LLM 을 사용하므로, 키 없이도 SSE 메시지 형식을 확인할 수 있습니다. 운영 팁:

- 프런트엔드는 `event:` 필드로 분기하면 충분합니다 (EventSource / SSE.js 모두 동일).
- 토큰 *조각* 이 그대로 흐르므로 unicode 경계가 깨질 수 있습니다 — 클라이언트에서 버퍼링한 뒤 렌더링하세요.
- reasoning 채널은 모델 내부 사고라 사용자에게 노출하지 말고 디버그용으로만 쓰세요.

---

## 체크포인트

- [ ] `loop_trace.json` — 기본 3건 + 안전장치 2건 = 5건이 모두 들어있다.
- [ ] `comparison.json` — `single`/`two_calls`의 통과율이 세 구현 모두 0.6 이상.
- [ ] `comparison.json` — `no_tool` 에서 세 구현 모두 `tool_invocations == 0`.
- [ ] `stream_trace.jsonl`의 줄 수 == 이벤트 총합, invariants 5개 모두 OK.

**다음:** Track 03 — Tools & MCP


## Wrap-up. 마무리

이 노트북에서는 같은 환율 문제를 손코딩 루프와 `ToolAgent`로 각각 풀어 보고, 마지막에는 같은 실행을 `run_stream` 이벤트와 SSE 메시지로 흘려보냈습니다.

이를 통해 에이전트 루프가 turn 예산, 도구 호출, 인자 파싱, 종료 조건으로 구성된다는 점과, `ToolAgent`가 그 반복 작업에 라우팅·최종 답 생성·trace 기록을 더해 운영하기 쉬운 형태로 감싸 준다는 점을 확인했습니다.
